# Brunel (2000) Balanced Network – A btorch Beginner Tutorial

This notebook walks through simulating the classic Brunel (2000) randomly-connected E/I spiking network with **btorch**.

## What you will learn
- The core btorch RSNN simulation pattern (`environ.context`, `init_net_state`, `reset_net`)
- How btorch manages **hidden state** via `register_memory`
- How to build **heterogeneous delays and weights** (Model B internals)
- How to run **memory-efficient long simulations** with chunking / CPU offloading
- How to manage configs with **OmegaConf dataclasses** and sweep parameters
- How to generate a **phase diagram** and overlay sampled points

Good reference: [Fabrizio Musacchio – The Brunel network](https://www.fabriziomusacchio.com/blog/2024-07-21-brunel_network)


---
## 1. What is the Brunel Network?

Brunel (2000) studied a large, sparsely-connected network of **excitatory (E)** and **inhibitory (I)** leaky integrate-and-fire (LIF) neurons driven by an external Poisson input.

- **Neurons**: 80 % excitatory, 20 % inhibitory.
- **Connectivity**: Random; each neuron receives $C_E$ E synapses and $C_I$ I synapses.
- **Weights**: E synapses are positive ($+J$); I synapses are negative ($-gJ$).
- **External drive**: Poisson spike trains with rate $
u_{ext}$.

The two free parameters that control the collective behavior are:
- **$g$** — relative inhibitory strength ($g = J_I / J_E$ in magnitude).
- **$\eta = \nu_{ext} / \nu_{thr}$** — external rate normalized by the threshold rate needed to reach firing threshold with E inputs alone.

Depending on $g$ and $\eta$ the network exhibits four qualitatively different regimes:

| Regime | Abbreviation | What it looks like | Typical metrics |
|--------|--------------|-------------------|----------------|
| **Synchronous Regular** | SR | Neurons fire in near-perfect lock-step | CV(ISI) ≈ 0, low frequency |
| **Asynchronous Irregular** | AI | Sparse, uncorrelated, Poisson-like firing | CV(ISI) ≈ 1, no dominant freq |
| **Synchronous Irregular (fast)** | SI-fast | Bursts of synchronous spikes at high frequency | CV(ISI) > 0.5, freq ≥ 40 Hz |
| **Synchronous Irregular (slow)** | SI-slow | Bursts of synchronous spikes at moderate frequency | CV(ISI) > 0.5, 10–40 Hz |

We will simulate each of these below and inspect their rasters and rates.


---
## 2. E/I Balance – why it matters

In the Brunel model the **net recurrent input** a neuron receives depends on the balance between excitation and inhibition.

- When $g$ is **small** (weak inhibition relative to excitation), excitation dominates and the network tends to **synchronize** (SR or SI).
- When $g$ is **large enough** (strong inhibition), excitatory and inhibitory currents roughly **cancel on average**. The network then operates in the **AI** regime: neurons fire irregularly and asynchronously because fluctuations—not the mean drive—push individual neurons over threshold.

The external drive $\eta$ also matters: at very low $\eta$ the network is nearly silent; at very high $\eta$ it can become super-synchronized even with strong inhibition.

This cancellation idea is central to the concept of **E/I balance** and is a cornerstone of computational neuroscience.


---
## 3. Environment and imports

We import btorch directly and set a deterministic seed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf

# btorch core
from btorch.models import environ, functional
from btorch.models.base import MemoryModule
from btorch.models.init import uniform_v_
from btorch.datasets.noise import poisson_noise

# Brunel package
from brunel2000.config import BrunelConfig, ModelAConfig, ModelBConfig, SimConfig
from brunel2000.connection import build_model_a_conn, build_model_b_conn
from brunel2000.simulate import run_simulation
from brunel2000.analysis import analyze
from brunel2000.sweep import run_one
from brunel2000.plot_phase_transition import classify_regime
from brunel2000.notebook_utils import plot_raster_and_rate, plot_raster_grid, plot_phase_map_overlay, metrics_table
from IPython.display import display

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


---
## 4. OmegaConf dataclass-first config basics

btorch examples use **dataclasses** as the single source of truth and merge CLI overrides with `OmegaConf`.
Below we create a small tutorial config with reduced neuron count so the notebook stays interactive.


In [ ]:
# Define a small default config inline
tutorial_cfg = BrunelConfig(
    model=ModelAConfig(
        n_neurons=2500,   # small for laptop/CPU
        c_e=200,
        c_ext=200,
        g=5.0,
        nu_ext_hz=20.0,
    ),
    sim=SimConfig(
        duration_ms=500.0,
        dt_ms=0.1,
        seed=SEED,
        device=device,
    ),
)

# OmegaConf merge pattern
defaults = OmegaConf.structured(tutorial_cfg)
overrides = OmegaConf.create({"model": {"g": 5.0, "nu_ext_hz": 20.0}})
merged = OmegaConf.unsafe_merge(defaults, overrides)
cfg = OmegaConf.to_object(merged)

print(type(cfg.model))
print(f"Neurons: {cfg.model.n_neurons}, g: {cfg.model.g}, nu_ext: {cfg.model.nu_ext_hz} Hz")
print(f"Duration: {cfg.sim.duration_ms} ms, dt: {cfg.sim.dt_ms} ms")


---
## 5. Basic simulation pattern (Model A)

The btorch simulation pattern is always the same three steps:
1. **Build** the model inside `environ.context(dt=...)`.
2. **Initialize** and **reset** network state (`init_net_state`, `reset_net`).
3. **Run** the forward pass.


In [ ]:
# ------------------------------------------------------------------
# 5a. Build the model inside dt context
# ------------------------------------------------------------------
dt = cfg.sim.dt_ms
T_steps = int(cfg.sim.duration_ms / dt)
batch = 1
n_e = int(cfg.model.n_neurons * cfg.model.n_e_ratio)

with environ.context(dt=dt):
    model = cfg.model.build_model(dt_ms=dt, device=device)

print(model)
print("State names tracked by RecurrentNN:", model.update_state_names)


In [ ]:
# ------------------------------------------------------------------
# 5b. Prepare Poisson external input
# ------------------------------------------------------------------
ext_rate = cfg.model.c_ext * cfg.model.nu_ext_hz
rng = torch.Generator(device=device)
rng.manual_seed(cfg.sim.seed)

counts = poisson_noise(
    batch,
    cfg.model.n_neurons,
    rate=ext_rate,
    T=T_steps,
    dt=dt / 1000.0,
    device=torch.device(device),
    dtype=torch.float32,
    generator=rng,
)
ext_input = counts * cfg.model.j
print(f"External input shape: {ext_input.shape}")  # (T, batch, n_neurons)


In [ ]:
# ------------------------------------------------------------------
# 5c. Init, reset, and run (the btorch pattern)
# ------------------------------------------------------------------
# explain, variables
functional.init_net_state(model, batch_size=batch, device=device)
functional.reset_net(model, batch_size=batch)
uniform_v_(model.neuron, set_reset_value=True)

# dt, numerical stability, and no_grad
with environ.context(dt=dt), torch.no_grad():
    spikes_rec, states = model(ext_input)

print(f"Spikes recorded shape: {spikes_rec.shape}")
print(f"Returned state keys: {list(states.keys())}")
print(f"Voltage shape: {states['neuron.v'].shape}")


In [ ]:
# ------------------------------------------------------------------
# 5d. Inspect a few neurons' voltage traces
# ------------------------------------------------------------------
voltages = states['neuron.v'][:, 0, :].detach().cpu().numpy()
t = np.arange(voltages.shape[0]) * dt

fig, ax = plt.subplots(figsize=(8, 3))
for nid in [0, n_e, n_e // 2]:
    ax.plot(t, voltages[:, nid], lw=0.8, label=f"neuron {nid}")
ax.axhline(cfg.model.theta, color='k', ls='--', lw=0.8, label='threshold')
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Voltage (mV)")
ax.legend(loc='upper right', fontsize=8)
ax.set_title("Model A – voltage traces (AI preset)")
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 5e. Raster + population rate (inline analysis)
# ------------------------------------------------------------------
spikes_np = spikes_rec[:, 0, :].detach().cpu().numpy()
fig, axes = plot_raster_and_rate(spikes_np, dt_ms=dt, n_e=n_e)
axes[0].set_title("Model A (g=5, eta≈1) – AI regime")
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 5f. Basic metrics inline
# ------------------------------------------------------------------
from brunel2000.simulate import SimulationResult
result = SimulationResult(
    spikes=spikes_rec,
    voltages=states['neuron.v'],
    psc=states.get('synapse.psc', torch.zeros_like(states['neuron.v'])),
    n_e=n_e,
    dt_ms=dt,
)
stats = analyze(result, model_type="ModelAConfig")
for k, v in stats.items():
    print(f"{k:25s}: {v:>10.3f}")


---
## 6. All four phases side-by-side

`BrunelConfig` provides convenient presets for the four regimes. We simulate each inline and display their rasters and metrics.


In [ ]:
regime_cases = {
    "sr": "model_a_sr",
    "ai": "model_a_ai",
    "si_fast": "model_a_si_fast",
    "si_slow": "model_a_si_slow",
}

results_for_grid = []
metrics_rows = []

for lbl, case_name in regime_cases.items():
    case_cfg = BrunelConfig.default_from_case(case_name)
    # keep small N/T for notebook speed
    case_cfg.model.n_neurons = 2500
    case_cfg.model.c_e = 200
    case_cfg.model.c_ext = 200
    case_cfg.sim.duration_ms = 500.0
    case_cfg.sim.dt_ms = 0.1
    case_cfg.sim.device = device

    with environ.context(dt=case_cfg.sim.dt_ms):
        mdl = case_cfg.model.build_model(dt_ms=case_cfg.sim.dt_ms, device=device)

    functional.init_net_state(mdl, batch_size=1, device=device)
    functional.reset_net(mdl, batch_size=1)
    uniform_v_(mdl.neuron, set_reset_value=True)

    dt_case = case_cfg.sim.dt_ms
    T_case = int(case_cfg.sim.duration_ms / dt_case)
    ext_rate_case = case_cfg.model.c_ext * case_cfg.model.nu_ext_hz
    rng_case = torch.Generator(device=device)
    rng_case.manual_seed(case_cfg.sim.seed)
    counts_case = poisson_noise(
        1, case_cfg.model.n_neurons, rate=ext_rate_case,
        T=T_case, dt=dt_case / 1000.0,
        device=torch.device(device), dtype=torch.float32, generator=rng_case,
    )
    ext_in = counts_case * case_cfg.model.j

    with environ.context(dt=dt_case), torch.no_grad():
        spk_case, st_case = mdl(ext_in)

    n_e_case = int(case_cfg.model.n_neurons * case_cfg.model.n_e_ratio)
    res_case = SimulationResult(
        spikes=spk_case,
        voltages=st_case['neuron.v'],
        psc=st_case.get('synapse.psc', torch.zeros_like(st_case['neuron.v'])),
        n_e=n_e_case,
        dt_ms=dt_case,
    )
    stt_case = analyze(res_case, model_type="ModelAConfig")
    stt_case['regime'] = lbl.upper()
    metrics_rows.append(stt_case)
    results_for_grid.append((spk_case[:, 0, :].detach().cpu().numpy(), dt_case, n_e_case, lbl.upper()))


In [ ]:
# ------------------------------------------------------------------
# 6b. Metrics table
# ------------------------------------------------------------------
df_metrics = metrics_table(metrics_rows)
df_metrics = df_metrics[['regime', 'mean_rate_overall_hz', 'mean_rate_e_hz', 'mean_rate_i_hz', 'cv_isi', 'dominant_freq_hz']]
display(df_metrics)


In [ ]:
# ------------------------------------------------------------------
# 6c. Raster grid
# ------------------------------------------------------------------
fig, axes = plot_raster_grid(results_for_grid, dt_ms=0.1, figsize=(10, 8))
fig.suptitle("Model A – Four regimes (rasters)", y=1.02, fontsize=14)
plt.show()


---
## 7. Hidden states and `register_memory`

btorch models are subclasses of `MemoryModule`. When you call `register_memory(name, value, shape)`, btorch:
- stores the tensor as a buffer,
- makes it persistent across forward calls,
- and tracks a **reset value** so `reset_net` can deterministically restore it.

### 7a. Minimal custom example


In [ ]:
class SimpleIntegrator(MemoryModule):
    def __init__(self, n_neuron):
        super().__init__()
        self.register_memory("v", 0.0, n_neuron)

    def forward(self, x):
        self.v = self.v + x
        return self.v

# Create and init
cell = SimpleIntegrator(5)
cell.init_state(batch_size=2)

x1 = torch.ones(2, 5)
out1 = cell(x1)
print("After first step:", cell.v)

x2 = torch.ones(2, 5) * 2.0
out2 = cell(x2)
print("After second step:", cell.v)

# Reset and observe deterministic restart
cell.reset()
print("After reset:", cell.v)

# Inspect via named_hidden_states
states = functional.named_hidden_states(cell)
print("Named hidden states:", states)


### 7b. Back to the Brunel model

We can inspect the hidden states and reset values of the full RSNN directly.


In [ ]:
# Rebuild the AI model to inspect its state machinery
ai_cfg = BrunelConfig.default_from_case("model_a_ai")
ai_cfg.model.n_neurons = 2500
ai_cfg.model.c_e = 200
ai_cfg.sim.duration_ms = 100.0
ai_cfg.sim.device = device

with environ.context(dt=ai_cfg.sim.dt_ms):
    ai_model = ai_cfg.model.build_model(dt_ms=ai_cfg.sim.dt_ms, device=device)

functional.init_net_state(ai_model, batch_size=1, device=device)
functional.reset_net(ai_model, batch_size=1)
uniform_v_(ai_model.neuron, set_reset_value=True)

# Inspect hidden states
hidden = functional.named_hidden_states(ai_model)
for k, v in hidden.items():
    print(f"{k:25s}: shape {tuple(v.shape)}, dtype {v.dtype}")

# Inspect reset values
rv = functional.named_memory_reset_values(ai_model)
for k, v in rv.items():
    print(f"{k:25s}: reset value type {type(v).__name__}")


---
## 8. Heterogeneous delays and weights (Model B)

Model B adds two forms of heterogeneity:
- **E/I-specific parameters** (different time constants, thresholds, synaptic weights).
- **Heterogeneous synaptic delays** drawn from a uniform distribution per connection type.

btorch handles this with two ingredients:
1. `make_hetersynapse_conn` — expands the sparse weight matrix to account for delay bins and receptor types.
2. `HeterSynapsePSC` — runs receptor-split synaptic dynamics with an internal `SpikeHistory`.


In [ ]:
# ------------------------------------------------------------------
# 8a. Build heterogeneous connectivity inline (small N for visibility)
# ------------------------------------------------------------------
n_e_small = 20
n_i_small = 5
c_e = 10
c_i = int(c_e * (1 - 0.8) / 0.8)  # preserve E/I fan-in ratio

conn_d, receptor_idx, delays_ms, n_delay_bins = build_model_b_conn(
    n_e=n_e_small,
    n_i=n_i_small,
    c_e=c_e,
    c_i=c_i,
    j_e=0.1,
    j_i=0.2,
    g_e=5.0,
    g_i=5.0,
    d_ee_ms=2.0,
    d_ei_ms=2.0,
    d_ie_ms=2.0,
    d_ii_ms=2.0,
    dt_ms=0.1,
    seed=SEED,
)

print(f"Delay-expanded matrix shape: {conn_d.shape}")
print(f"Number of receptor columns: {len(receptor_idx)}")
print(f"Number of delay bins:      {n_delay_bins}")
print(f"Delay range (ms):          {delays_ms.min():.2f} – {delays_ms.max():.2f}")

# Show a snippet of receptor_idx
display(receptor_idx.head(6))


In [ ]:
# ------------------------------------------------------------------
# 8b. Histogram of sampled delays
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(delays_ms, bins=20, color='steelblue', edgecolor='white')
ax.set_xlabel("Synaptic delay (ms)")
ax.set_ylabel("Count")
ax.set_title("Model B – sampled heterogeneous delays")
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 8c. Build Model B RSNN inline (small N, short run)
# ------------------------------------------------------------------
from btorch.models.linear import SparseConn
from btorch.models.neurons import LIF
from btorch.models.synapse import ExponentialPSC, HeterSynapsePSC
from btorch.models.rnn import RecurrentNN

n_neuron_b = n_e_small + n_i_small
n_receptor = len(receptor_idx)

with environ.context(dt=0.1):
    linear_b = SparseConn(conn_d, enforce_dale=False)

    # Neuron params per population
    tau = torch.full((n_neuron_b,), 20.0)
    tau[n_e_small:] = 10.0
    v_threshold = torch.full((n_neuron_b,), 20.0)
    v_reset = torch.full((n_neuron_b,), 10.0)
    tau_ref = torch.full((n_neuron_b,), 2.0)
    c_m = torch.ones(n_neuron_b)

    neuron_b = LIF(
        n_neuron=n_neuron_b,
        tau=tau,
        tau_ref=tau_ref,
        v_threshold=v_threshold,
        v_reset=v_reset,
        c_m=c_m,
    )

    # Synaptic time constants per receptor
    tau_syn_2d = torch.full((n_neuron_b, n_receptor), 0.3)
    post_types = receptor_idx['post_receptor_type'].values
    i_cols_i = [i for i, pt in enumerate(post_types) if pt == 'I']
    if i_cols_i:
        tau_syn_2d[:, i_cols_i] = 0.5
    tau_syn_flat = tau_syn_2d.T.flatten()

    synapse_b = HeterSynapsePSC(
        n_neuron=n_neuron_b,
        n_receptor=n_receptor,
        receptor_type_index=receptor_idx,
        linear=linear_b,
        base_psc=ExponentialPSC,
        tau_syn=tau_syn_flat,
        max_delay_steps=n_delay_bins,
        use_circular_buffer=True,
    )

    model_b = RecurrentNN(
        neuron=neuron_b,
        synapse=synapse_b,
        step_mode='m',
        update_state_names=('neuron.v', 'synapse.psc'),
    )
    model_b.to(device)

print(model_b)


In [ ]:
# ------------------------------------------------------------------
# 8d. Simulate Model B inline and inspect E/I rates
# ------------------------------------------------------------------
T_b = 3000  # 300 ms
batch_b = 1
rate_e_ext = 200 * 15.0  # c_ext * nu_e_ext_hz
rate_i_ext = 200 * 15.0

rng_b = torch.Generator(device=device)
rng_b.manual_seed(SEED)

counts_e_b = poisson_noise(
    batch_b, n_e_small, rate=rate_e_ext, T=T_b, dt=0.1 / 1000.0,
    device=torch.device(device), dtype=torch.float32, generator=rng_b,
)
counts_i_b = poisson_noise(
    batch_b, n_i_small, rate=rate_i_ext, T=T_b, dt=0.1 / 1000.0,
    device=torch.device(device), dtype=torch.float32, generator=rng_b,
)
ext_input_b = torch.cat([counts_e_b * 0.1, counts_i_b * 0.2], dim=-1)

functional.init_net_state(model_b, batch_size=batch_b, device=device)
functional.reset_net(model_b, batch_size=batch_b)
uniform_v_(model_b.neuron, set_reset_value=True)

with environ.context(dt=0.1), torch.no_grad():
    spikes_b, states_b = model_b(ext_input_b)

# Compute E/I rates inline
spikes_b_np = spikes_b[:, 0, :].detach().cpu().numpy()
bin_ms = 1.0
bin_steps = max(int(bin_ms / 0.1), 1)
T_tot_b = spikes_b_np.shape[0]
bins_b = np.arange(0, T_tot_b + bin_steps, bin_steps)
rate_e = np.empty(len(bins_b) - 1)
rate_i = np.empty(len(bins_b) - 1)
t_centers_b = np.empty(len(bins_b) - 1)
for k in range(len(bins_b) - 1):
    lo, hi = bins_b[k], bins_b[k + 1]
    rate_e[k] = spikes_b_np[lo:hi, :n_e_small].sum() / (n_e_small * bin_ms * 1e-3)
    rate_i[k] = spikes_b_np[lo:hi, n_e_small:].sum() / (n_i_small * bin_ms * 1e-3)
    t_centers_b[k] = (lo + hi) / 2 * 0.1

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(t_centers_b, rate_e, color='#54a24b', lw=1.0, label='E')
ax.plot(t_centers_b, rate_i, color='#e45756', lw=1.0, label='I')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Rate (Hz)')
ax.legend(loc='upper right')
ax.set_title('Model B – E/I population rates (heterogeneous delays + weights)')
plt.tight_layout()
plt.show()


---
## 9. Memory-efficient simulation with chunking

For very long simulations, keeping the entire sequence on GPU can run out of memory. btorch’s `RecurrentNN` supports **chunked computation** with optional CPU offloading:

- `chunk_size` splits the sequence into blocks processed one at a time.
- `cpu_offload=True` moves outputs and states to CPU between chunks, freeing GPU memory.

**Guardrail**: `chunk_size` must be a multiple of `unroll` (default `unroll=1`).


In [ ]:
# ------------------------------------------------------------------
# 9a. Build a model with large T (still small N so it fits)
# ------------------------------------------------------------------
chunk_cfg = BrunelConfig.default_from_case("model_a_ai")
chunk_cfg.model.n_neurons = 1200
chunk_cfg.model.c_e = 120
chunk_cfg.sim.duration_ms = 1000.0   # 2 seconds = lots of steps
chunk_cfg.sim.dt_ms = 0.1
chunk_cfg.sim.device = device

with environ.context(dt=chunk_cfg.sim.dt_ms):
    chunk_model = chunk_cfg.model.build_model(dt_ms=chunk_cfg.sim.dt_ms, device=device)

dt_c = chunk_cfg.sim.dt_ms
T_c = int(chunk_cfg.sim.duration_ms / dt_c)
ext_rate_c = chunk_cfg.model.c_ext * chunk_cfg.model.nu_ext_hz
rng_c = torch.Generator(device=device)
rng_c.manual_seed(chunk_cfg.sim.seed)
counts_c = poisson_noise(
    1, chunk_cfg.model.n_neurons, rate=ext_rate_c, T=T_c, dt=dt_c / 1000.0,
    device=torch.device(device), dtype=torch.float32, generator=rng_c,
)
ext_in_c = counts_c * chunk_cfg.model.j
print(f"Sequence length: {T_c} steps, input shape: {ext_in_c.shape}")


In [ ]:
# ------------------------------------------------------------------
# 9b. Run without chunking
# ------------------------------------------------------------------
import time

functional.init_net_state(chunk_model, batch_size=1, device=device)
functional.reset_net(chunk_model, batch_size=1)
uniform_v_(chunk_model.neuron, set_reset_value=True)

if device.startswith("cuda"):
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
with environ.context(dt=dt_c), torch.no_grad():
    spk_full, st_full = chunk_model(ext_in_c)
t_full = time.time() - t0

mem_full = 0.0
if device.startswith("cuda"):
    mem_full = torch.cuda.max_memory_allocated() / 1024**2
    torch.cuda.empty_cache()

print(f"No chunking: {t_full:.3f} s, peak memory: {mem_full:.1f} MiB")


In [ ]:
# ------------------------------------------------------------------
# 9c. Run WITH chunking + CPU offloading
# ------------------------------------------------------------------
# Must reset state before second run
functional.init_net_state(chunk_model, batch_size=1, device=device)
functional.reset_net(chunk_model, batch_size=1)
uniform_v_(chunk_model.neuron, set_reset_value=True)

chunk_model.chunk_size = 256   # process 256 steps at a time
chunk_model.cpu_offload = True

if device.startswith("cuda"):
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
with environ.context(dt=dt_c), torch.no_grad():
    spk_chunk, st_chunk = chunk_model(ext_in_c)
t_chunk = time.time() - t0

mem_chunk = 0.0
if device.startswith("cuda"):
    mem_chunk = torch.cuda.max_memory_allocated() / 1024**2
    torch.cuda.empty_cache()

print(f"Chunked (256): {t_chunk:.3f} s, peak memory: {mem_chunk:.1f} MiB")

if device.startswith("cuda"):
    print(f"Memory reduction: {mem_full - mem_chunk:.1f} MiB ({100*(1-mem_chunk/mem_full):.1f}%)")
else:
    print("(CPU mode: timing comparison only)")

# Sanity-check: outputs should be nearly identical
max_diff = (spk_full.cpu() - spk_chunk.cpu()).abs().max().item()
print(f"Max spike difference: {max_diff:.6f}")


---
## 10. OmegaConf small sweep + phase plot

We now do a tiny parameter sweep over $(g, \eta)$ and overlay sampled points on a coarse regime map. This mirrors the SLURM workflow shown in `run_phase_sweep.slurm`, but entirely inside the notebook.


In [ ]:
# ------------------------------------------------------------------
# 10a. Build a tiny sweep config inline
# ------------------------------------------------------------------
from brunel2000.sweep import SweepConfig, SweepGridConfig, SweepSimConfig, SweepExecConfig

sweep_cfg = SweepConfig(
    model='a',
    grid=SweepGridConfig(g_min=1.0, g_max=8.0, g_steps=8, eta_min=0.0, eta_max=4.0, eta_steps=5),
    sim=SweepSimConfig(
        n_neurons=1200,
        c_e=120,
        duration_ms=300.0,
        dt_ms=0.1,
        n_trials=1,
        device='cpu',
    ),
    exec=SweepExecConfig(output_dir='outputs/tutorial_sweep', num_workers=1),
)

gs = np.linspace(sweep_cfg.grid.g_min, sweep_cfg.grid.g_max, sweep_cfg.grid.g_steps)
etas = np.linspace(sweep_cfg.grid.eta_min, sweep_cfg.grid.eta_max, sweep_cfg.grid.eta_steps)
print(f"Sweep grid: {len(gs)} g points x {len(etas)} eta points = {len(gs)*len(etas)} simulations")


In [ ]:
# ------------------------------------------------------------------
# 10b. Run sweep inline (single-threaded)
# ------------------------------------------------------------------
sweep_rows = []
point_idx = 0
for eta in etas:
    for g in gs:
        trial_idx = 0
        row = run_one(
            model_kind='a',
            g=float(g),
            eta=float(eta),
            trial_idx=trial_idx,
            j_i_override=None,
            point_idx=point_idx,
            cfg=sweep_cfg,
        )
        row['status'] = 'ok'
        sweep_rows.append(row)
        point_idx += 1

df_sweep = pd.DataFrame(sweep_rows)
print(f"Completed {len(df_sweep)} points. Columns:", list(df_sweep.columns))


In [ ]:
# ------------------------------------------------------------------
# 10c. Classify regimes and plot phase map
# ------------------------------------------------------------------
df_sweep['regime'] = df_sweep.apply(classify_regime, axis=1)

# Optional: overlay a few sampled points (e.g. every 3rd point)
overlay = df_sweep.iloc[::3].copy()

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_phase_map_overlay(df_sweep, model='a', overlay_df=overlay, ax=ax)
ax.set_title('Model A – Phase diagram (g vs `eta`)')
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 10d. Also show rate and CV as heatmaps
# ------------------------------------------------------------------
piv_rate = df_sweep.pivot_table(index='eta', columns='g', values='mean_rate_overall_hz', aggfunc='mean').sort_index()
piv_cv = df_sweep.pivot_table(index='eta', columns='g', values='cv_isi', aggfunc='mean').sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, piv, label in [(axes[0], piv_rate, 'Mean rate (Hz)'), (axes[1], piv_cv, 'CV(ISI)')]:
    x = np.asarray(piv.columns, dtype=float)
    y = np.asarray(piv.index, dtype=float)
    extent = [x.min(), x.max(), y.min(), y.max()]
    im = ax.imshow(piv.values, origin='lower', aspect='auto', extent=extent, cmap='viridis')
    fig.colorbar(im, ax=ax, shrink=0.9)
    ax.set_xlabel('g')
    ax.set_ylabel(r'$\eta = \nu_{ext} / \nu_{thr}$')
    ax.set_title(label)

fig.tight_layout()
plt.show()


---
## 11. Recap and suggested experiments

**Core btorch patterns covered**
- `environ.context(dt=...)` wraps every simulation.
- `init_net_state(..., batch_size=...)` + `reset_net(...)` before each forward pass.
- `register_memory` makes state persistent and resettable.
- `named_hidden_states` and `named_memory_reset_values` inspect state machinery.
- `HeterSynapsePSC` + `make_hetersynapse_conn` handle heterogeneity.
- `chunk_size` + `cpu_offload` keep long simulations memory-efficient.
- OmegaConf dataclasses make configs reproducible and sweepable.

**Try next**
1. Change `g` and `nu_ext_hz` in the one-shot Model A cell and watch the raster change.
2. Increase neurons to `10000` and set `chunk_size=512` to see memory savings on GPU.
3. Modify delays in Model B (`d_ee_ms`, etc.) and observe how rate transients change.
4. Run a sweep for Model B by setting `sweep_cfg.model = 'b'`.
5. Use `functional.named_hidden_states(model)` mid-simulation to inspect how `neuron.v` evolves.
6. Try different `tau_ms` values (membrane time constant) and note how CV(ISI) shifts.
